# Reproduce the Eight-Gaussians diffusion results


This notebook reproduces the Eight-Gaussians diffusion figures and tables in the paper. Run generating cells to rebuild results; each saved artifact is loaded in the following cell. Results are written under `res/8gauss/`. Generation scripts use the configs in `exp/configs/`.


In [ ]:
import math
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from omegaconf import OmegaConf

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "exp/reproduce_diffusion_8gauss.ipynb").is_file()
)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
sys.path.insert(1, str(ROOT / "src" / "engression"))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_ROOT = ROOT / "models"


In [ ]:
from utils.eight_gauss import (
    eight_gaussians,
    eight_gaussians_centers,
    empirical_mode_weights_nearest_center,
    energy_distance_unbiased,
)
from utils.experiment_io import (
    save_torch_atomic,
    set_reproducible_seed,
    write_csv_atomic,
    write_json_atomic,
)
from utils.noise_geometry import (
    generalized_gaussian_actual_rms,
    generalized_gaussian_rms_constant,
    reference_rho_to_sigma,
    sample_generalized_gaussian_noise,
    spd_2d_from_uv,
)
from utils.sampling import (
    annealed_langevin_generate_parameter_path,
    load_parameter_conditioned_engressor,
    mc_score_from_posterior_samples_spd,
)


# Five-parameter conditioned model


Two independently trained Engressors, with training seeds 42 and 43, provide the score, generation, and marginal-tracking comparisons.


In [ ]:
FIVE_PARAM_DEVICE = device
FIVE_PARAM_MODEL_ROOT = MODEL_ROOT / "score_and_diffusion"
FIVE_PARAM_OUT_DIR = Path("res/8gauss/diffusion/full5_spd")
FIVE_PARAM_RUN_DIR = FIVE_PARAM_OUT_DIR / "runs"
FIVE_PARAM_OUT_DIR.mkdir(parents=True, exist_ok=True)
FIVE_PARAM_RUN_DIR.mkdir(parents=True, exist_ok=True)

FIVE_PARAM_MODEL_SPECS = [
    {
        "training_seed": 42,
        "run_name": "full_spd_seed42",
        "checkpoint": "model.pt",
    },
    {
        "training_seed": 43,
        "run_name": "full_spd_seed43",
        "checkpoint": "model.pt",
    },
]
FIVE_PARAM_FIGURE_MODEL_SEED = 43
FIVE_PARAM_SCORE_SEED = 104729
FIVE_PARAM_SCORE_GRID_SIZE = 33
FIVE_PARAM_SCORE_MC = 256
FIVE_PARAM_SCORE_LIMIT = 3.5
FIVE_PARAM_DATA_RADIUS = 2.0
FIVE_PARAM_STD_CLEAN = 0.2
FIVE_PARAM_PLOT_RC = {
    "figure.dpi": 160,
    "savefig.dpi": 300,
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 8,
    "lines.linewidth": 1.2,
    "axes.linewidth": 0.8,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.03,
}

for spec in FIVE_PARAM_MODEL_SPECS:
    model_dir = FIVE_PARAM_MODEL_ROOT / spec["run_name"]
    config_path = model_dir / "config.yaml"
    checkpoint_path = model_dir / spec["checkpoint"]
    if not config_path.is_file() or not checkpoint_path.is_file():
        raise FileNotFoundError(f"Missing config or checkpoint in {model_dir}")
    config = OmegaConf.load(config_path)
    if str(config.noise.condition) != "full_spd":
        raise ValueError(f"{model_dir} is not a five-parameter model")


In [ ]:
def five_param_subsample(samples, max_points, seed):
    samples = torch.as_tensor(samples).detach().cpu()
    max_points = min(int(max_points), samples.shape[0])
    if max_points == samples.shape[0]:
        return samples
    generator = torch.Generator().manual_seed(int(seed))
    indices = torch.randperm(samples.shape[0], generator=generator)[:max_points]
    return samples[indices]


def five_param_stratified_subsample(batches, max_points, seed):
    batches = [torch.as_tensor(batch).detach().cpu() for batch in batches]
    total_points = sum(len(batch) for batch in batches)
    if total_points <= int(max_points):
        return torch.cat(batches, dim=0)
    points_per_batch, remainder = divmod(int(max_points), len(batches))
    selected = []
    for batch_index, batch in enumerate(batches):
        n_selected = points_per_batch + int(batch_index < remainder)
        generator = torch.Generator().manual_seed(int(seed) + batch_index)
        indices = torch.randperm(len(batch), generator=generator)[:n_selected]
        selected.append(batch[indices])
    return torch.cat(selected, dim=0)


def five_param_endpoint_metrics(generated_batches, reference, *, subsample, seed):
    if isinstance(generated_batches, torch.Tensor):
        generated_batches = [generated_batches]
    generated_batches = [
        torch.as_tensor(batch).detach().cpu() for batch in generated_batches
    ]
    generated_all = torch.cat(generated_batches, dim=0)
    generated_eval = five_param_stratified_subsample(
        generated_batches,
        subsample,
        seed,
    )
    reference_eval = five_param_subsample(reference, subsample, seed + 1)
    mode_weights = empirical_mode_weights_nearest_center(
        generated_all,
        radius=FIVE_PARAM_DATA_RADIUS,
    )[0]
    target_weights = torch.full_like(mode_weights, 1.0 / mode_weights.numel())
    return {
        "signed_energy_distance": float(
            energy_distance_unbiased(generated_eval, reference_eval)
        ),
        "mode_weight_tv": float(0.5 * (mode_weights - target_weights).abs().sum()),
        "minimum_mode_mass": float(mode_weights.min()),
        "modes_below_one_percent": int((mode_weights < 0.01).sum()),
        "mean_radial_error": float(
            (torch.linalg.vector_norm(generated_all, dim=1) - FIVE_PARAM_DATA_RADIUS)
            .abs()
            .mean()
        ),
    }


def five_param_score_at_condition(model_info, condition, grid, *, n_mc, seed):
    beta = float(condition["beta"])
    lam = float(condition["lam"])
    rho = float(condition["rho"])
    uv = torch.tensor(
        [condition["u"], condition["v"]],
        device=grid.device,
        dtype=grid.dtype,
    )
    sigma = float(reference_rho_to_sigma(rho, beta, 2))
    set_reproducible_seed(seed)
    with torch.inference_mode():
        posterior = model_info["sampler"].sample(
            grid,
            beta,
            lam,
            rho,
            uv,
            int(n_mc),
        )
        score = mc_score_from_posterior_samples_spd(
            y=grid,
            x_samples=posterior,
            sigma=sigma,
            beta=beta,
            lam=lam,
            Sigma=spd_2d_from_uv(uv),
        )
    grid_size = int(round(math.sqrt(grid.shape[0])))
    if grid_size**2 != grid.shape[0]:
        raise ValueError("score plotting grid must be square")
    return {
        **condition,
        "sigma": sigma,
        "actual_rms": float(generalized_gaussian_actual_rms(rho, beta, lam)),
        "score": score.detach().cpu().numpy().reshape(grid_size, grid_size, 2),
    }


def five_param_plot_score_sweep(results, titles, file_stem):
    log_magnitudes = [
        np.log1p(np.linalg.norm(result["score"], axis=-1)) for result in results
    ]
    color_min = min(float(values.min()) for values in log_magnitudes)
    color_max = max(float(values.max()) for values in log_magnitudes)
    color_max = max(color_max, color_min + 1e-12)
    color_levels = np.linspace(color_min, color_max, 31)

    with plt.rc_context(FIVE_PARAM_PLOT_RC):
        fig, axes = plt.subplots(
            1,
            len(results),
            figsize=(2.55 * len(results) + 0.55, 2.85),
            dpi=300,
            constrained_layout=True,
        )
        axes = np.atleast_1d(axes)
        for axis, result, title, log_magnitude in zip(
            axes,
            results,
            titles,
            log_magnitudes,
        ):
            contour = axis.contourf(
                five_param_score_xx_np,
                five_param_score_yy_np,
                log_magnitude,
                levels=color_levels,
                cmap="viridis",
            )
            magnitude = np.linalg.norm(result["score"], axis=-1)
            direction = result["score"] / (magnitude[..., None] + 1e-12)
            axis.streamplot(
                five_param_score_xx_np[0],
                five_param_score_yy_np[:, 0],
                direction[..., 0],
                direction[..., 1],
                color="white",
                density=1.15,
                linewidth=0.65,
                arrowsize=0.7,
            )
            axis.scatter(
                five_param_centers[:, 0],
                five_param_centers[:, 1],
                marker="x",
                s=18,
                color="white",
                linewidths=0.8,
                zorder=6,
            )
            axis.set_xlim(-FIVE_PARAM_SCORE_LIMIT, FIVE_PARAM_SCORE_LIMIT)
            axis.set_ylim(-FIVE_PARAM_SCORE_LIMIT, FIVE_PARAM_SCORE_LIMIT)
            axis.set_aspect("equal")
            axis.set_xlabel("$y_1$")
            axis.set_ylabel("$y_2$")
            axis.set_title(title)

        colorbar = fig.colorbar(contour, ax=axes.tolist(), shrink=0.82, pad=0.02)
        colorbar.set_label(r"$\log(1+\|\widehat s_m(y)\|)$")
        fig.savefig(FIVE_PARAM_OUT_DIR / f"{file_stem}.png", bbox_inches="tight")
        fig.savefig(FIVE_PARAM_OUT_DIR / f"{file_stem}.pdf", bbox_inches="tight")
    return fig


## Score fields over the five parameter sweeps


In [ ]:
FIVE_PARAM_BASE_CONDITION = {
    "beta": 1.4,
    "lam": 1.8,
    "rho": 0.570322,
    "u": 0.0,
    "v": 0.6,
}
FIVE_PARAM_SWEEP_VALUES = {
    "beta": [1.3, 1.5, 1.75, 2.0],
    "lambda": [1.0, 1.4, 1.8, 2.5],
    "rho": [0.126738, 0.316846, 0.570322, 0.950537],
    "u": [-0.6, -0.3, 0.0, 0.3, 0.6],
    "v": [-0.6, -0.3, 0.0, 0.3, 0.6],
}
five_param_sweep_conditions = {}

for parameter_name, values in FIVE_PARAM_SWEEP_VALUES.items():
    conditions = []
    for value in values:
        condition = dict(FIVE_PARAM_BASE_CONDITION)
        if parameter_name == "lambda":
            condition["lam"] = float(value)
        elif parameter_name == "u":
            condition["u"] = float(value)
            condition["v"] = 0.0
        elif parameter_name == "v":
            condition["u"] = 0.0
            condition["v"] = float(value)
        else:
            condition[parameter_name] = float(value)
        condition["factor"] = parameter_name
        condition["factor_value"] = float(value)
        conditions.append(condition)
    five_param_sweep_conditions[parameter_name] = conditions

five_param_score_axis = torch.linspace(
    -FIVE_PARAM_SCORE_LIMIT,
    FIVE_PARAM_SCORE_LIMIT,
    FIVE_PARAM_SCORE_GRID_SIZE,
    device=FIVE_PARAM_DEVICE,
)
five_param_score_yy, five_param_score_xx = torch.meshgrid(
    five_param_score_axis,
    five_param_score_axis,
    indexing="ij",
)
five_param_score_grid = torch.stack(
    [five_param_score_xx.reshape(-1), five_param_score_yy.reshape(-1)],
    dim=1,
)
five_param_score_xx_np = five_param_score_xx.detach().cpu().numpy()
five_param_score_yy_np = five_param_score_yy.detach().cpu().numpy()
five_param_centers = eight_gaussians_centers(FIVE_PARAM_DATA_RADIUS).cpu().numpy()
five_param_figure_spec = next(
    spec
    for spec in FIVE_PARAM_MODEL_SPECS
    if int(spec["training_seed"]) == FIVE_PARAM_FIGURE_MODEL_SEED
)
five_param_figure_model = {
    **load_parameter_conditioned_engressor(
        FIVE_PARAM_MODEL_ROOT / five_param_figure_spec["run_name"],
        checkpoint=five_param_figure_spec["checkpoint"],
        device=FIVE_PARAM_DEVICE,
    ),
    "spec": five_param_figure_spec,
}
five_param_streamplot_results = {}

for parameter_name, conditions in five_param_sweep_conditions.items():
    five_param_streamplot_results[parameter_name] = [
        five_param_score_at_condition(
            five_param_figure_model,
            condition,
            five_param_score_grid,
            n_mc=FIVE_PARAM_SCORE_MC,
            seed=FIVE_PARAM_SCORE_SEED,
        )
        for condition in conditions
    ]

{name: len(results) for name, results in five_param_streamplot_results.items()}
save_torch_atomic(
    five_param_streamplot_results, FIVE_PARAM_OUT_DIR / "score_field_sweeps.pt"
)


In [ ]:
five_param_streamplot_results = torch.load(
    FIVE_PARAM_OUT_DIR / "score_field_sweeps.pt", map_location="cpu", weights_only=False
)


In [ ]:
five_param_beta_titles = [
    rf"$\beta={result['beta']:.2f}$" for result in five_param_streamplot_results["beta"]
]
fig = five_param_plot_score_sweep(
    five_param_streamplot_results["beta"],
    five_param_beta_titles,
    "full5_spd_score_fields_beta_sweep",
)


In [ ]:
five_param_lambda_titles = [
    rf"$\lambda={result['lam']:.2f}$"
    for result in five_param_streamplot_results["lambda"]
]
fig = five_param_plot_score_sweep(
    five_param_streamplot_results["lambda"],
    five_param_lambda_titles,
    "full5_spd_score_fields_lambda_sweep",
)


In [ ]:
five_param_rho_titles = [
    rf"$\rho={result['rho']:.3f}$" for result in five_param_streamplot_results["rho"]
]
fig = five_param_plot_score_sweep(
    five_param_streamplot_results["rho"],
    five_param_rho_titles,
    "full5_spd_score_fields_rho_sweep",
)


In [ ]:
five_param_u_titles = [
    rf"$(u,v)=({result['u']:.1f},0)$" for result in five_param_streamplot_results["u"]
]
fig = five_param_plot_score_sweep(
    five_param_streamplot_results["u"],
    five_param_u_titles,
    "full5_spd_score_fields_u_sweep",
)


In [ ]:
five_param_v_titles = [
    rf"$(u,v)=(0,{result['v']:.1f})$" for result in five_param_streamplot_results["v"]
]
fig = five_param_plot_score_sweep(
    five_param_streamplot_results["v"],
    five_param_v_titles,
    "full5_spd_score_fields_v_sweep",
)


In [ ]:
FIVE_PARAM_MAIN_PANEL_SELECTION = {
    "beta": 2.0,
    "lambda": 2.5,
    "rho": 0.950537,
    "u": 0.6,
    "v": -0.6,
}
FIVE_PARAM_MAIN_PANEL_TITLES = {
    "beta": lambda result: rf"$\beta={result['beta']:.2f}$",
    "lambda": lambda result: rf"$\lambda={result['lam']:.2f}$",
    "rho": lambda result: rf"$\rho={result['rho']:.3f}$",
    "u": lambda result: rf"$(u,v)=({result['u']:.1f},0)$",
    "v": lambda result: rf"$(u,v)=(0,{result['v']:.1f})$",
}

five_param_main_panel_results = []
five_param_main_panel_titles = []
for parameter_name, selected_value in FIVE_PARAM_MAIN_PANEL_SELECTION.items():
    result = next(
        result
        for result in five_param_streamplot_results[parameter_name]
        if math.isclose(result["factor_value"], selected_value)
    )
    five_param_main_panel_results.append(result)
    five_param_main_panel_titles.append(
        FIVE_PARAM_MAIN_PANEL_TITLES[parameter_name](result)
    )

fig = five_param_plot_score_sweep(
    five_param_main_panel_results,
    five_param_main_panel_titles,
    "full5_spd_score_fields_main_panel",
)
fig.set_size_inches(12.0, 2.8)
fig.set_layout_engine("compressed", w_pad=0.02, h_pad=0.02, wspace=0.015)
five_param_main_axes = fig.axes[: len(five_param_main_panel_results)]
for column_index, axis in enumerate(five_param_main_axes):
    if column_index > 0:
        axis.sharey(five_param_main_axes[0])
        axis.set_ylabel("")
        axis.tick_params(axis="y", left=False, labelleft=False)
    else:
        axis.set_ylabel("$y_2$", fontsize=14)
    axis.set_xlabel("$y_1$", fontsize=14, labelpad=2)
    axis.set_title(axis.get_title(), fontsize=15, pad=6)
    axis.tick_params(axis="both", labelsize=12, pad=2)

five_param_main_colorbar_axis = fig.axes[-1]
five_param_main_colorbar_axis.tick_params(labelsize=12)
five_param_main_colorbar_axis.yaxis.label.set_size(14)
for extension in ("png", "pdf"):
    fig.savefig(
        FIVE_PARAM_OUT_DIR / f"full5_spd_score_fields_main_panel.{extension}",
        bbox_inches="tight",
        pad_inches=0.03,
    )


## Four noise-parameter paths


In [ ]:
FIVE_PARAM_DIFFUSION_N_LEVELS = 51
FIVE_PARAM_DIFFUSION_N_SAMPLES = 4096
FIVE_PARAM_DIFFUSION_N_MC = 64
FIVE_PARAM_DIFFUSION_N_STEPS = 10
FIVE_PARAM_DIFFUSION_ENERGY_SUBSAMPLE = 4096
FIVE_PARAM_DIFFUSION_ALPHA_CAP = 0.025
FIVE_PARAM_DIFFUSION_UPDATE_CLIP = 1000.0
FIVE_PARAM_DIFFUSION_SEEDS = [2718, 3141, 5772]
FIVE_PARAM_DIFFUSION_REFERENCE_SEED = 299792
FIVE_PARAM_DIFFUSION_EVALUATION_SEED = 323846
FIVE_PARAM_CLEAN_BASELINE_SEEDS = [81173, 104729]
FIVE_PARAM_FIGURE_SAMPLING_SEED = 2718
FIVE_PARAM_DIFFUSION_METRICS = [
    "signed_energy_distance",
    "mode_weight_tv",
    "minimum_mode_mass",
    "modes_below_one_percent",
    "mean_radial_error",
]
FIVE_PARAM_PATH_LABELS = {
    "isotropic_fixed_tail": "Isotropic, fixed tail",
    "fixed_non_diagonal_fixed_tail": "Fixed non-diagonal, fixed tail",
    "rotating_fixed_tail": "Rotating covariance, fixed tail",
    "joint_five_parameter": "Joint five-parameter path",
}
FIVE_PARAM_PATH_COLORS = {
    "isotropic_fixed_tail": "#5f6368",
    "fixed_non_diagonal_fixed_tail": "#2f6fb0",
    "rotating_fixed_tail": "#3c8d5a",
    "joint_five_parameter": "#c25a37",
}

five_param_diffusion_progress = torch.linspace(
    0.0,
    1.0,
    FIVE_PARAM_DIFFUSION_N_LEVELS,
    dtype=torch.float64,
)
five_param_fixed_beta = torch.full_like(five_param_diffusion_progress, 1.4)
five_param_fixed_lambda = torch.full_like(five_param_diffusion_progress, 1.8)
five_param_sigma_reference = torch.as_tensor(
    np.geomspace(1.0, 0.01, FIVE_PARAM_DIFFUSION_N_LEVELS),
    dtype=torch.float64,
)
five_param_rho_reference = (
    five_param_sigma_reference
    * generalized_gaussian_rms_constant(
        1.4,
        2,
        dtype=torch.float64,
    )
)
five_param_actual_rms_reference = generalized_gaussian_actual_rms(
    five_param_rho_reference,
    five_param_fixed_beta,
    five_param_fixed_lambda,
)
five_param_uv_isotropic = torch.zeros(
    FIVE_PARAM_DIFFUSION_N_LEVELS,
    2,
    dtype=torch.float64,
)
five_param_uv_fixed = torch.tensor(
    [0.0, 0.6],
    dtype=torch.float64,
).repeat(FIVE_PARAM_DIFFUSION_N_LEVELS, 1)
five_param_path_angle = math.pi / 2.0 + 2.0 * math.pi * five_param_diffusion_progress
five_param_uv_rotating = 0.6 * torch.stack(
    [five_param_path_angle.cos(), five_param_path_angle.sin()],
    dim=1,
)
five_param_uv_rotating[0] = torch.tensor([0.0, 0.6], dtype=torch.float64)
five_param_uv_rotating[-1] = torch.tensor([0.0, 0.6], dtype=torch.float64)

five_param_joint_beta = (
    1.4 + 0.5 * torch.sin(math.pi * five_param_diffusion_progress).square()
)
five_param_joint_lambda = 1.8 + 0.4 * torch.sin(
    2.0 * math.pi * five_param_diffusion_progress
)
five_param_joint_rho = five_param_actual_rms_reference * five_param_joint_lambda.pow(
    1.0 / five_param_joint_beta
)
five_param_joint_beta[[0, -1]] = five_param_fixed_beta[[0, -1]]
five_param_joint_lambda[[0, -1]] = five_param_fixed_lambda[[0, -1]]
five_param_joint_rho[[0, -1]] = five_param_rho_reference[[0, -1]]

five_param_diffusion_schedules = {
    "isotropic_fixed_tail": {
        "beta": five_param_fixed_beta,
        "lam": five_param_fixed_lambda,
        "rho": five_param_rho_reference,
        "uv": five_param_uv_isotropic,
        "actual_rms": five_param_actual_rms_reference,
    },
    "fixed_non_diagonal_fixed_tail": {
        "beta": five_param_fixed_beta,
        "lam": five_param_fixed_lambda,
        "rho": five_param_rho_reference,
        "uv": five_param_uv_fixed,
        "actual_rms": five_param_actual_rms_reference,
    },
    "rotating_fixed_tail": {
        "beta": five_param_fixed_beta,
        "lam": five_param_fixed_lambda,
        "rho": five_param_rho_reference,
        "uv": five_param_uv_rotating,
        "actual_rms": five_param_actual_rms_reference,
    },
    "joint_five_parameter": {
        "beta": five_param_joint_beta,
        "lam": five_param_joint_lambda,
        "rho": five_param_joint_rho,
        "uv": five_param_uv_rotating,
        "actual_rms": five_param_actual_rms_reference,
    },
}

for path_name, schedule in five_param_diffusion_schedules.items():
    spd_2d_from_uv(schedule["uv"])
    realized_rms = generalized_gaussian_actual_rms(
        schedule["rho"],
        schedule["beta"],
        schedule["lam"],
    )
    torch.testing.assert_close(realized_rms, schedule["actual_rms"])
    if torch.any(realized_rms[1:] > realized_rms[:-1] + 1e-12):
        raise ValueError(f"{path_name} does not decrease in actual RMS")

torch.testing.assert_close(
    five_param_diffusion_schedules["rotating_fixed_tail"]["beta"][[0, -1]],
    five_param_diffusion_schedules["joint_five_parameter"]["beta"][[0, -1]],
)
torch.testing.assert_close(
    five_param_diffusion_schedules["rotating_fixed_tail"]["lam"][[0, -1]],
    five_param_diffusion_schedules["joint_five_parameter"]["lam"][[0, -1]],
)
torch.testing.assert_close(
    five_param_diffusion_schedules["rotating_fixed_tail"]["rho"][[0, -1]],
    five_param_diffusion_schedules["joint_five_parameter"]["rho"][[0, -1]],
)

five_param_schedule_rows = []
for path_name, schedule in five_param_diffusion_schedules.items():
    sigma_values = reference_rho_to_sigma(
        schedule["rho"],
        schedule["beta"],
        2,
    )
    for level in range(FIVE_PARAM_DIFFUSION_N_LEVELS):
        five_param_schedule_rows.append(
            {
                "path": path_name,
                "level": level,
                "progress": float(five_param_diffusion_progress[level]),
                "beta": float(schedule["beta"][level]),
                "lambda": float(schedule["lam"][level]),
                "rho": float(schedule["rho"][level]),
                "sigma": float(sigma_values[level]),
                "actual_rms": float(schedule["actual_rms"][level]),
                "u": float(schedule["uv"][level, 0]),
                "v": float(schedule["uv"][level, 1]),
            }
        )

five_param_schedule_df = pd.DataFrame(five_param_schedule_rows)
five_param_schedule_df["uv_radius"] = np.sqrt(
    five_param_schedule_df["u"] ** 2 + five_param_schedule_df["v"] ** 2
)
write_csv_atomic(
    five_param_schedule_df,
    FIVE_PARAM_OUT_DIR / "diffusion_realized_schedules.csv",
)
five_param_schedule_summary_df = (
    five_param_schedule_df.groupby("path")
    .agg(
        beta_min=("beta", "min"),
        beta_max=("beta", "max"),
        lambda_min=("lambda", "min"),
        lambda_max=("lambda", "max"),
        rho_start=("rho", "first"),
        rho_end=("rho", "last"),
        actual_rms_start=("actual_rms", "first"),
        actual_rms_end=("actual_rms", "last"),
        uv_radius_max=("uv_radius", "max"),
    )
    .reset_index()
)
write_csv_atomic(
    five_param_schedule_summary_df,
    FIVE_PARAM_OUT_DIR / "diffusion_schedule_summary.csv",
)


In [ ]:
five_param_schedule_df = pd.read_csv(
    FIVE_PARAM_OUT_DIR / "diffusion_realized_schedules.csv"
)
five_param_schedule_summary_df = pd.read_csv(
    FIVE_PARAM_OUT_DIR / "diffusion_schedule_summary.csv"
)


In [ ]:
def five_param_diffusion_bundle_path(model_seed, path_name, sampling_seed):
    return FIVE_PARAM_RUN_DIR / (
        f"model_s{int(model_seed)}__{path_name}__sampling_s{int(sampling_seed)}.pt"
    )


def five_param_diffusion_metadata(spec, path_name, sampling_seed):
    schedule = five_param_diffusion_schedules[path_name]
    return {
        "model_seed": int(spec["training_seed"]),
        "run_name": spec["run_name"],
        "checkpoint": spec["checkpoint"],
        "path": path_name,
        "sampling_seed": int(sampling_seed),
        "n_samples": FIVE_PARAM_DIFFUSION_N_SAMPLES,
        "n_mc": FIVE_PARAM_DIFFUSION_N_MC,
        "n_steps_each": FIVE_PARAM_DIFFUSION_N_STEPS,
        "alpha_cap": FIVE_PARAM_DIFFUSION_ALPHA_CAP,
        "update_clip": FIVE_PARAM_DIFFUSION_UPDATE_CLIP,
        "beta": schedule["beta"].tolist(),
        "lambda": schedule["lam"].tolist(),
        "rho": schedule["rho"].tolist(),
        "actual_rms": schedule["actual_rms"].tolist(),
        "uv": schedule["uv"].tolist(),
    }


five_param_diffusion_config = {
    "models": FIVE_PARAM_MODEL_SPECS,
    "paths": FIVE_PARAM_PATH_LABELS,
    "sampling_seeds": FIVE_PARAM_DIFFUSION_SEEDS,
    "n_levels": FIVE_PARAM_DIFFUSION_N_LEVELS,
    "n_samples": FIVE_PARAM_DIFFUSION_N_SAMPLES,
    "n_mc": FIVE_PARAM_DIFFUSION_N_MC,
    "n_steps_each": FIVE_PARAM_DIFFUSION_N_STEPS,
    "energy_subsample": FIVE_PARAM_DIFFUSION_ENERGY_SUBSAMPLE,
    "reference_seed": FIVE_PARAM_DIFFUSION_REFERENCE_SEED,
    "evaluation_seed": FIVE_PARAM_DIFFUSION_EVALUATION_SEED,
    "clean_baseline_seeds": FIVE_PARAM_CLEAN_BASELINE_SEEDS,
    "endpoint_pooling": "pool sampling seeds within each trained model",
    "uncertainty": "mean plus or minus sample SD across trained-model seeds",
    "alpha_cap": FIVE_PARAM_DIFFUSION_ALPHA_CAP,
    "update_clip": FIVE_PARAM_DIFFUSION_UPDATE_CLIP,
}
write_json_atomic(
    five_param_diffusion_config,
    FIVE_PARAM_OUT_DIR / "diffusion_config.json",
)
five_param_diffusion_config


In [ ]:
five_param_diffusion_models = [
    {
        **load_parameter_conditioned_engressor(
            FIVE_PARAM_MODEL_ROOT / spec["run_name"],
            checkpoint=spec["checkpoint"],
            device=FIVE_PARAM_DEVICE,
        ),
        "spec": spec,
    }
    for spec in FIVE_PARAM_MODEL_SPECS
]
five_param_initial_states = {}

for model_info in five_param_diffusion_models:
    spec = model_info["spec"]
    model_seed = int(spec["training_seed"])
    sampler = model_info["sampler"]

    for path_name, schedule_cpu in five_param_diffusion_schedules.items():
        schedule_device = {
            name: values.to(
                device=FIVE_PARAM_DEVICE,
                dtype=torch.get_default_dtype(),
            )
            for name, values in schedule_cpu.items()
        }
        for sampling_seed in FIVE_PARAM_DIFFUSION_SEEDS:
            bundle_path = five_param_diffusion_bundle_path(
                model_seed,
                path_name,
                sampling_seed,
            )
            expected_metadata = five_param_diffusion_metadata(
                spec,
                path_name,
                sampling_seed,
            )
            payload = None
            if bundle_path.exists():
                candidate = torch.load(
                    bundle_path, map_location="cpu", weights_only=False
                )
                if candidate.get("metadata") == expected_metadata:
                    payload = candidate

            if payload is None:
                set_reproducible_seed(sampling_seed)
                generation = annealed_langevin_generate_parameter_path(
                    sampler.sample,
                    n_samples=FIVE_PARAM_DIFFUSION_N_SAMPLES,
                    dim=2,
                    schedule=schedule_device,
                    n_steps_each=FIVE_PARAM_DIFFUSION_N_STEPS,
                    n_mc=FIVE_PARAM_DIFFUSION_N_MC,
                    alpha_cap=FIVE_PARAM_DIFFUSION_ALPHA_CAP,
                    update_clip=FIVE_PARAM_DIFFUSION_UPDATE_CLIP,
                    state_clip=None,
                    device=FIVE_PARAM_DEVICE,
                    verbose=False,
                    show_progress=False,
                )
                payload = {
                    "metadata": expected_metadata,
                    "generated": generation["final"],
                    "final_noisy": generation["last_noisy"],
                    "trajectory": generation["trajectory"],
                    "diagnostics": generation["diagnostics"],
                }

            save_torch_atomic(payload, bundle_path)
            five_param_initial_states[(model_seed, sampling_seed, path_name)] = payload[
                "trajectory"
            ][0]
            print(f"model={model_seed} path={path_name} sampling_seed={sampling_seed}")

for spec in FIVE_PARAM_MODEL_SPECS:
    model_seed = int(spec["training_seed"])
    for sampling_seed in FIVE_PARAM_DIFFUSION_SEEDS:
        fixed_initial = five_param_initial_states[
            (model_seed, sampling_seed, "fixed_non_diagonal_fixed_tail")
        ]
        rotating_initial = five_param_initial_states[
            (model_seed, sampling_seed, "rotating_fixed_tail")
        ]
        joint_initial = five_param_initial_states[
            (model_seed, sampling_seed, "joint_five_parameter")
        ]
        if not torch.equal(fixed_initial, rotating_initial):
            raise AssertionError("fixed and rotating paths do not share initialization")
        if not torch.equal(rotating_initial, joint_initial):
            raise AssertionError("rotating and joint paths do not share initialization")

len(five_param_initial_states)


In [ ]:
five_param_diffusion_bundles = {}
five_param_diffusion_rows = []
five_param_initial_states = {}

set_reproducible_seed(FIVE_PARAM_DIFFUSION_REFERENCE_SEED)
five_param_diffusion_reference = eight_gaussians(
    FIVE_PARAM_DIFFUSION_ENERGY_SUBSAMPLE,
    radius=FIVE_PARAM_DATA_RADIUS,
    std=FIVE_PARAM_STD_CLEAN,
    device="cpu",
)

for spec in FIVE_PARAM_MODEL_SPECS:
    model_seed = int(spec["training_seed"])
    for path_name in FIVE_PARAM_PATH_LABELS:
        for sampling_seed in FIVE_PARAM_DIFFUSION_SEEDS:
            bundle_path = five_param_diffusion_bundle_path(
                model_seed,
                path_name,
                sampling_seed,
            )
            if not bundle_path.exists():
                raise FileNotFoundError(bundle_path)
            payload = torch.load(bundle_path, map_location="cpu", weights_only=False)
            expected_metadata = five_param_diffusion_metadata(
                spec,
                path_name,
                sampling_seed,
            )
            if payload.get("metadata") != expected_metadata:
                raise ValueError(f"stale diffusion bundle: {bundle_path}")
            generated = torch.as_tensor(payload["generated"]).detach().cpu()
            if generated.shape != (FIVE_PARAM_DIFFUSION_N_SAMPLES, 2):
                raise ValueError(f"unexpected generated shape in {bundle_path}")
            if len(payload["trajectory"]) != FIVE_PARAM_DIFFUSION_N_LEVELS + 1:
                raise ValueError(f"unexpected trajectory length in {bundle_path}")
            bundle_key = (model_seed, path_name, int(sampling_seed))
            five_param_diffusion_bundles[bundle_key] = payload
            five_param_initial_states[bundle_key] = payload["trajectory"][0]
            metrics = five_param_endpoint_metrics(
                generated,
                five_param_diffusion_reference,
                subsample=FIVE_PARAM_DIFFUSION_ENERGY_SUBSAMPLE,
                seed=FIVE_PARAM_DIFFUSION_EVALUATION_SEED,
            )
            five_param_diffusion_rows.append(
                {
                    "model_seed": model_seed,
                    "run_name": spec["run_name"],
                    "path": path_name,
                    "sampling_seed": int(sampling_seed),
                    "n_samples": len(generated),
                    "reference_seed": FIVE_PARAM_DIFFUSION_REFERENCE_SEED,
                    "bundle_path": str(bundle_path),
                    **metrics,
                }
            )

for spec in FIVE_PARAM_MODEL_SPECS:
    model_seed = int(spec["training_seed"])
    for sampling_seed in FIVE_PARAM_DIFFUSION_SEEDS:
        fixed_initial = five_param_initial_states[
            (model_seed, "fixed_non_diagonal_fixed_tail", sampling_seed)
        ]
        rotating_initial = five_param_initial_states[
            (model_seed, "rotating_fixed_tail", sampling_seed)
        ]
        joint_initial = five_param_initial_states[
            (model_seed, "joint_five_parameter", sampling_seed)
        ]
        if not torch.equal(fixed_initial, rotating_initial):
            raise AssertionError("fixed and rotating paths do not share initialization")
        if not torch.equal(rotating_initial, joint_initial):
            raise AssertionError("rotating and joint paths do not share initialization")

five_param_diffusion_raw_df = pd.DataFrame(five_param_diffusion_rows).sort_values(
    ["path", "model_seed", "sampling_seed"]
)
five_param_diffusion_per_model_rows = []
for spec in FIVE_PARAM_MODEL_SPECS:
    model_seed = int(spec["training_seed"])
    for path_name in FIVE_PARAM_PATH_LABELS:
        generated_batches = [
            five_param_diffusion_bundles[(model_seed, path_name, sampling_seed)][
                "generated"
            ]
            for sampling_seed in FIVE_PARAM_DIFFUSION_SEEDS
        ]
        metrics = five_param_endpoint_metrics(
            generated_batches,
            five_param_diffusion_reference,
            subsample=FIVE_PARAM_DIFFUSION_ENERGY_SUBSAMPLE,
            seed=FIVE_PARAM_DIFFUSION_EVALUATION_SEED,
        )
        five_param_diffusion_per_model_rows.append(
            {
                "path": path_name,
                "model_seed": model_seed,
                "run_name": spec["run_name"],
                "n_sampling_seeds": len(FIVE_PARAM_DIFFUSION_SEEDS),
                "n_generated_samples": sum(len(batch) for batch in generated_batches),
                **metrics,
            }
        )

five_param_diffusion_per_model_df = pd.DataFrame(
    five_param_diffusion_per_model_rows
).sort_values(["path", "model_seed"])
five_param_diffusion_summary_df = five_param_diffusion_per_model_df.groupby("path")[
    FIVE_PARAM_DIFFUSION_METRICS
].agg(["mean", "std"])
five_param_diffusion_summary_df.columns = [
    f"{metric}_{statistic}"
    for metric, statistic in five_param_diffusion_summary_df.columns.to_flat_index()
]
five_param_diffusion_summary_df = five_param_diffusion_summary_df.reset_index()
five_param_diffusion_summary_df["n_training_seeds"] = len(FIVE_PARAM_MODEL_SPECS)
five_param_diffusion_summary_df["sampling_seeds_per_model"] = len(
    FIVE_PARAM_DIFFUSION_SEEDS
)
five_param_diffusion_summary_df["generated_samples_per_model"] = (
    FIVE_PARAM_DIFFUSION_N_SAMPLES * len(FIVE_PARAM_DIFFUSION_SEEDS)
)

five_param_clean_baseline_rows = []
for baseline_seed in FIVE_PARAM_CLEAN_BASELINE_SEEDS:
    clean_batches = []
    for batch_index in range(len(FIVE_PARAM_DIFFUSION_SEEDS)):
        set_reproducible_seed(int(baseline_seed) + batch_index)
        clean_batches.append(
            eight_gaussians(
                FIVE_PARAM_DIFFUSION_N_SAMPLES,
                radius=FIVE_PARAM_DATA_RADIUS,
                std=FIVE_PARAM_STD_CLEAN,
                device="cpu",
            )
        )
    metrics = five_param_endpoint_metrics(
        clean_batches,
        five_param_diffusion_reference,
        subsample=FIVE_PARAM_DIFFUSION_ENERGY_SUBSAMPLE,
        seed=FIVE_PARAM_DIFFUSION_EVALUATION_SEED,
    )
    five_param_clean_baseline_rows.append(
        {
            "baseline_seed": int(baseline_seed),
            "n_sampling_batches": len(clean_batches),
            "n_samples": sum(len(batch) for batch in clean_batches),
            **metrics,
        }
    )
five_param_clean_baseline_df = pd.DataFrame(five_param_clean_baseline_rows)

five_param_joint_vs_rotating_df = five_param_diffusion_per_model_df[
    five_param_diffusion_per_model_df["path"] == "joint_five_parameter"
].merge(
    five_param_diffusion_per_model_df[
        five_param_diffusion_per_model_df["path"] == "rotating_fixed_tail"
    ],
    on=["model_seed", "run_name"],
    suffixes=("_joint", "_rotating"),
)
five_param_joint_vs_rotating_df = five_param_joint_vs_rotating_df[
    ["model_seed", "run_name"]
].copy()
for metric in FIVE_PARAM_DIFFUSION_METRICS:
    joint_values = five_param_diffusion_per_model_df[
        five_param_diffusion_per_model_df["path"] == "joint_five_parameter"
    ].set_index("model_seed")[metric]
    rotating_values = five_param_diffusion_per_model_df[
        five_param_diffusion_per_model_df["path"] == "rotating_fixed_tail"
    ].set_index("model_seed")[metric]
    five_param_joint_vs_rotating_df[
        f"{metric}_difference"
    ] = five_param_joint_vs_rotating_df["model_seed"].map(
        joint_values
    ) - five_param_joint_vs_rotating_df[
        "model_seed"
    ].map(
        rotating_values
    )

five_param_diffusion_protocol = (
    "Two independently trained five-parameter posterior models; four paths "
    "sharing one actual-RMS schedule and sampler budget; three paired "
    f"sampling seeds with {FIVE_PARAM_DIFFUSION_N_SAMPLES} final samples per "
    "seed. Sampling batches are pooled within each trained model before "
    "nonlinear endpoint metrics are computed. Mode coverage and radial error "
    "use all 12,288 pooled samples; signed energy distance uses a fixed "
    "seed-stratified 4,096-sample subset and one fixed independent 4,096-point "
    "clean reference. The table reports mean ± sample SD across the two "
    "trained models. Every generated point is included in the coverage and "
    "radial metrics without selection."
)

write_csv_atomic(
    five_param_diffusion_raw_df,
    FIVE_PARAM_OUT_DIR / "diffusion_metrics_raw.csv",
)
write_csv_atomic(
    five_param_diffusion_per_model_df,
    FIVE_PARAM_OUT_DIR / "diffusion_metrics_per_model.csv",
)
write_csv_atomic(
    five_param_diffusion_summary_df,
    FIVE_PARAM_OUT_DIR / "diffusion_metrics_summary.csv",
)
write_csv_atomic(
    five_param_clean_baseline_df,
    FIVE_PARAM_OUT_DIR / "diffusion_clean_baseline.csv",
)
write_csv_atomic(
    five_param_joint_vs_rotating_df,
    FIVE_PARAM_OUT_DIR / "diffusion_joint_vs_rotating.csv",
)
write_json_atomic(
    {
        **five_param_diffusion_config,
        "generated_samples_per_model_path": (
            FIVE_PARAM_DIFFUSION_N_SAMPLES * len(FIVE_PARAM_DIFFUSION_SEEDS)
        ),
        "coverage_and_radial_sample_size": (
            FIVE_PARAM_DIFFUSION_N_SAMPLES * len(FIVE_PARAM_DIFFUSION_SEEDS)
        ),
        "energy_distance_sample_size": FIVE_PARAM_DIFFUSION_ENERGY_SUBSAMPLE,
        "protocol": five_param_diffusion_protocol,
    },
    FIVE_PARAM_OUT_DIR / "diffusion_protocol.json",
)
save_torch_atomic(
    {
        "config": five_param_diffusion_config,
        "bundle_paths": five_param_diffusion_raw_df["bundle_path"].tolist(),
        "raw_metrics": five_param_diffusion_raw_df.to_dict(orient="records"),
        "per_model_metrics": five_param_diffusion_per_model_df.to_dict(
            orient="records"
        ),
        "summary": five_param_diffusion_summary_df.to_dict(orient="records"),
        "clean_baseline": five_param_clean_baseline_df.to_dict(orient="records"),
        "joint_vs_rotating": five_param_joint_vs_rotating_df.to_dict(orient="records"),
    },
    FIVE_PARAM_OUT_DIR / "diffusion_result_index.pt",
)
five_param_diffusion_summary_df


In [ ]:
five_param_diffusion_bundles = {
    (int(spec["training_seed"]), path_name, int(sampling_seed)): torch.load(
        five_param_diffusion_bundle_path(
            int(spec["training_seed"]), path_name, int(sampling_seed)
        ),
        map_location="cpu",
        weights_only=False,
    )
    for spec in FIVE_PARAM_MODEL_SPECS
    for path_name in FIVE_PARAM_PATH_LABELS
    for sampling_seed in FIVE_PARAM_DIFFUSION_SEEDS
}
five_param_diffusion_raw_df = pd.read_csv(
    FIVE_PARAM_OUT_DIR / "diffusion_metrics_raw.csv"
)
five_param_diffusion_per_model_df = pd.read_csv(
    FIVE_PARAM_OUT_DIR / "diffusion_metrics_per_model.csv"
)
five_param_diffusion_summary_df = pd.read_csv(
    FIVE_PARAM_OUT_DIR / "diffusion_metrics_summary.csv"
)
five_param_clean_baseline_df = pd.read_csv(
    FIVE_PARAM_OUT_DIR / "diffusion_clean_baseline.csv"
)
five_param_joint_vs_rotating_df = pd.read_csv(
    FIVE_PARAM_OUT_DIR / "diffusion_joint_vs_rotating.csv"
)


In [ ]:
five_param_diffusion_summary_rows = []
for path_name in FIVE_PARAM_PATH_LABELS:
    row = five_param_diffusion_summary_df[
        five_param_diffusion_summary_df["path"] == path_name
    ].iloc[0]
    five_param_diffusion_summary_rows.append(
        {
            "Path": FIVE_PARAM_PATH_LABELS[path_name],
            "Training seeds": int(row["n_training_seeds"]),
            "Sampling runs/model": int(row["sampling_seeds_per_model"]),
            "Generated samples/model": int(row["generated_samples_per_model"]),
            "Signed energy distance": (
                f"{row['signed_energy_distance_mean']:.5f} ± "
                f"{row['signed_energy_distance_std']:.5f}"
            ),
            "Mode-weight TV": (
                f"{row['mode_weight_tv_mean']:.4f} ± "
                f"{row['mode_weight_tv_std']:.4f}"
            ),
            "Minimum mode mass": (
                f"{row['minimum_mode_mass_mean']:.4f} ± "
                f"{row['minimum_mode_mass_std']:.4f}"
            ),
            "Modes below 1%": (
                f"{row['modes_below_one_percent_mean']:.2f} ± "
                f"{row['modes_below_one_percent_std']:.2f}"
            ),
            "Mean radial error": (
                f"{row['mean_radial_error_mean']:.4f} ± "
                f"{row['mean_radial_error_std']:.4f}"
            ),
        }
    )

five_param_diffusion_summary_table = pd.DataFrame(five_param_diffusion_summary_rows)
write_csv_atomic(
    five_param_diffusion_summary_table,
    FIVE_PARAM_OUT_DIR / "diffusion_summary_table.csv",
)


In [ ]:
five_param_diffusion_summary_table = pd.read_csv(
    FIVE_PARAM_OUT_DIR / "diffusion_summary_table.csv"
)
five_param_diffusion_summary_table


In [ ]:
five_param_clean_baseline_summary = five_param_clean_baseline_df[
    FIVE_PARAM_DIFFUSION_METRICS
].agg(["mean", "std"])
five_param_clean_baseline_table = pd.DataFrame(
    [
        {
            "Reference": "Independent clean 8Gauss draws",
            "Replicate groups": len(FIVE_PARAM_CLEAN_BASELINE_SEEDS),
            "Samples/replicate": int(five_param_clean_baseline_df["n_samples"].iloc[0]),
            "Signed energy distance": (
                f"{five_param_clean_baseline_summary.loc['mean', 'signed_energy_distance']:.5f} ± "
                f"{five_param_clean_baseline_summary.loc['std', 'signed_energy_distance']:.5f}"
            ),
            "Mode-weight TV": (
                f"{five_param_clean_baseline_summary.loc['mean', 'mode_weight_tv']:.4f} ± "
                f"{five_param_clean_baseline_summary.loc['std', 'mode_weight_tv']:.4f}"
            ),
            "Minimum mode mass": (
                f"{five_param_clean_baseline_summary.loc['mean', 'minimum_mode_mass']:.4f} ± "
                f"{five_param_clean_baseline_summary.loc['std', 'minimum_mode_mass']:.4f}"
            ),
            "Modes below 1%": (
                f"{five_param_clean_baseline_summary.loc['mean', 'modes_below_one_percent']:.2f} ± "
                f"{five_param_clean_baseline_summary.loc['std', 'modes_below_one_percent']:.2f}"
            ),
            "Mean radial error": (
                f"{five_param_clean_baseline_summary.loc['mean', 'mean_radial_error']:.4f} ± "
                f"{five_param_clean_baseline_summary.loc['std', 'mean_radial_error']:.4f}"
            ),
        }
    ]
)
write_csv_atomic(
    five_param_clean_baseline_table,
    FIVE_PARAM_OUT_DIR / "diffusion_clean_baseline_table.csv",
)


In [ ]:
five_param_clean_baseline_table = pd.read_csv(
    FIVE_PARAM_OUT_DIR / "diffusion_clean_baseline_table.csv"
)
five_param_clean_baseline_table


In [ ]:
five_param_joint_vs_rotating_labels = {
    "signed_energy_distance": "Signed energy distance",
    "mode_weight_tv": "Mode-weight TV",
    "minimum_mode_mass": "Minimum mode mass",
    "modes_below_one_percent": "Modes below 1%",
    "mean_radial_error": "Mean radial error",
}
five_param_joint_vs_rotating_rows = []
for metric, label in five_param_joint_vs_rotating_labels.items():
    differences = five_param_joint_vs_rotating_df[f"{metric}_difference"]
    digits = 5 if metric == "signed_energy_distance" else 4
    if metric == "modes_below_one_percent":
        digits = 2
    five_param_joint_vs_rotating_rows.append(
        {
            "Metric (joint minus rotating)": label,
            "Training seeds": len(differences),
            "Mean ± SD": (
                f"{differences.mean():.{digits}f} ± "
                f"{differences.std(ddof=1):.{digits}f}"
            ),
        }
    )
five_param_joint_vs_rotating_table = pd.DataFrame(five_param_joint_vs_rotating_rows)
write_csv_atomic(
    five_param_joint_vs_rotating_table,
    FIVE_PARAM_OUT_DIR / "diffusion_joint_vs_rotating_table.csv",
)


In [ ]:
five_param_joint_vs_rotating_table = pd.read_csv(
    FIVE_PARAM_OUT_DIR / "diffusion_joint_vs_rotating_table.csv"
)
five_param_joint_vs_rotating_table


## Fixed-parameter path isolation


Generate the 11 fixed-parameter arms with the sampling script; it saves one bundle per arm, model, and sampling seed. To run it in the background instead, use `bash exp/sample_diffusion_8gauss_controls.sh`. The following cells evaluate the saved bundles with the same metrics as the manuscript.

In [ ]:
controls_run = subprocess.run(
    [
        sys.executable,
        str(ROOT / "exp/sample_diffusion_8gauss_controls.py"),
        "--config",
        "exp/configs/diffusion-8gauss-controls.yaml",
        "--device",
        str(device),
        "--all",
    ],
    cwd=ROOT,
    check=True,
)

In [ ]:
CETERIS_CONFIG_PATH = Path("exp/configs/diffusion-8gauss-controls.yaml")
ceteris_source_config = OmegaConf.load(CETERIS_CONFIG_PATH)
OmegaConf.resolve(ceteris_source_config)
CETERIS_RESOLVED_CONFIG = OmegaConf.to_container(ceteris_source_config, resolve=True)
CETERIS_SAMPLER_CONFIG = OmegaConf.to_container(
    ceteris_source_config.sampler, resolve=True
)
CETERIS_OUT_DIR = Path(str(ceteris_source_config.output_root))
CETERIS_RUN_DIR = CETERIS_OUT_DIR / "runs"
CETERIS_ARMS_PATH = Path(str(ceteris_source_config.arms_file))
CETERIS_CONFIG_ARTIFACT_PATH = CETERIS_OUT_DIR / "config.json"
CETERIS_RAW_PATH = CETERIS_OUT_DIR / "raw.csv"
CETERIS_TRACKING_RAW_PATH = CETERIS_OUT_DIR / "tracking_raw.csv"
CETERIS_PER_MODEL_PATH = CETERIS_OUT_DIR / "per_model.csv"
CETERIS_SUMMARY_PATH = CETERIS_OUT_DIR / "summary.csv"
CETERIS_TABLE_PATH = CETERIS_OUT_DIR / "table.csv"
CETERIS_FIGURE_PATH = CETERIS_OUT_DIR / "ceteris_paribus_metrics.png"
CETERIS_FIGURE_PDF_PATH = CETERIS_OUT_DIR / "ceteris_paribus_metrics.pdf"
CETERIS_PROTOCOL_PATH = CETERIS_OUT_DIR / "evaluation_protocol.json"
ceteris_arms_df = pd.read_csv(CETERIS_ARMS_PATH, sep="\t")
ceteris_arms_df.insert(0, "arm_index", np.arange(len(ceteris_arms_df)))
CETERIS_MODEL_SPECS = [
    {
        "training_seed": int(spec.training_seed),
        "run_name": str(spec.run_name),
        "checkpoint": str(spec.checkpoint),
    }
    for spec in ceteris_source_config.models
]
CETERIS_SAMPLING_SEEDS = [int(seed) for seed in ceteris_source_config.sampling_seeds]
CETERIS_FINALIZATION_SEED_OFFSET = int(ceteris_source_config.finalization.seed_offset)
CETERIS_TRACKING_LEVELS = [
    int(level) for level in ceteris_source_config.tracking.levels
]
CETERIS_BURN_IN_EXCLUDED_LEVELS = {
    int(level) for level in ceteris_source_config.tracking.burn_in_excluded_levels
}
CETERIS_ENDPOINT_METRICS = list(FIVE_PARAM_DIFFUSION_METRICS)
CETERIS_SUMMARY_METRICS = [*CETERIS_ENDPOINT_METRICS, "post_burnin_tracking_max"]
CETERIS_EXPECTED_BUNDLE_COUNT = (
    len(ceteris_arms_df) * len(CETERIS_MODEL_SPECS) * len(CETERIS_SAMPLING_SEEDS)
)
ceteris_expected_arms = [
    {
        "arm_index": int(arm["arm_index"]),
        "arm_id": str(arm["arm_id"]),
        "parameter": str(arm["parameter"]),
        "value": str(arm["value"]),
        "value_label": str(arm["value_label"]),
        "beta": float(arm["beta"]),
        "lam": float(arm["lam"]),
        "u": float(arm["u"]),
        "v": float(arm["v"]),
        "plot_order": int(arm["plot_order"]),
    }
    for arm in ceteris_arms_df.to_dict(orient="records")
]
ceteris_result_models = CETERIS_MODEL_SPECS


def ceteris_bundle_path(arm_id, model_seed, sampling_seed):
    return (
        CETERIS_RUN_DIR
        / str(arm_id)
        / f"model_s{int(model_seed)}"
        / f"sampling_s{int(sampling_seed)}.pt"
    )


def ceteris_exact_noisy_marginal_samples(arm, schedule, level, seed):
    beta = float(schedule["beta"][level])
    lam = float(schedule["lam"][level])
    rho = float(schedule["rho"][level])
    uv = torch.tensor(
        [float(arm["u"]), float(arm["v"])], dtype=torch.get_default_dtype()
    )
    Sigma = spd_2d_from_uv(uv)
    sigma = float(reference_rho_to_sigma(rho, beta, 2))
    set_reproducible_seed(seed)
    clean = eight_gaussians(
        int(ceteris_source_config.tracking.n_reference),
        radius=float(ceteris_source_config.data.radius),
        std=float(ceteris_source_config.data.std_clean),
        device="cpu",
    )
    noise = sample_generalized_gaussian_noise(
        len(clean),
        beta=beta,
        lam=lam,
        Sigma=Sigma,
        sigma=sigma,
        device="cpu",
        dtype=clean.dtype,
    )
    return clean + noise


def ceteris_mean_sd_entry(row, metric, digits):
    mean = float(row[f"{metric}_mean"])
    sd = float(row[f"{metric}_std"])
    if not np.isfinite(mean):
        return ""
    return f"{mean:.{digits}f} ± {sd:.{digits}f}"


In [ ]:
ceteris_arm_records = ceteris_expected_arms
ceteris_expected_bundle_paths = [
    ceteris_bundle_path(arm["arm_id"], spec["training_seed"], sampling_seed)
    for arm in ceteris_arm_records
    for spec in CETERIS_MODEL_SPECS
    for sampling_seed in CETERIS_SAMPLING_SEEDS
]
if len(ceteris_expected_bundle_paths) != CETERIS_EXPECTED_BUNDLE_COUNT:
    raise ValueError("Ceteris-paribus bundle design has the wrong size")
ceteris_missing_bundle_paths = [
    path for path in ceteris_expected_bundle_paths if not path.exists()
]
if ceteris_missing_bundle_paths:
    n_present = CETERIS_EXPECTED_BUNDLE_COUNT - len(ceteris_missing_bundle_paths)
    raise FileNotFoundError(
        f"Incomplete ceteris-paribus campaign: found {n_present} of {CETERIS_EXPECTED_BUNDLE_COUNT} bundles; missing {[str(path) for path in ceteris_missing_bundle_paths]}"
    )
ceteris_bundles = {}
ceteris_schedules = {}
ceteris_raw_rows = []
set_reproducible_seed(int(ceteris_source_config.evaluation.reference_seed))
ceteris_clean_reference = eight_gaussians(
    int(ceteris_source_config.evaluation.energy_subsample),
    radius=float(ceteris_source_config.data.radius),
    std=float(ceteris_source_config.data.std_clean),
    device="cpu",
)
for arm in ceteris_arm_records:
    arm_id = str(arm["arm_id"])
    for spec in CETERIS_MODEL_SPECS:
        model_seed = int(spec["training_seed"])
        for sampling_seed in CETERIS_SAMPLING_SEEDS:
            bundle_path = ceteris_bundle_path(arm_id, model_seed, sampling_seed)
            payload = torch.load(bundle_path, map_location="cpu", weights_only=False)
            generated = torch.as_tensor(payload["final"]).detach().cpu()
            schedule = payload["metadata"]["schedule"]
            if arm_id in ceteris_schedules:
                if ceteris_schedules[arm_id] != schedule:
                    raise ValueError(f"Schedules disagree for arm {arm_id}")
            else:
                ceteris_schedules[arm_id] = schedule
            ceteris_bundles[arm_id, model_seed, sampling_seed] = payload
            metrics = five_param_endpoint_metrics(
                generated,
                ceteris_clean_reference,
                subsample=int(ceteris_source_config.evaluation.energy_subsample),
                seed=int(ceteris_source_config.evaluation.evaluation_seed),
            )
            ceteris_raw_rows.append(
                {
                    "record_type": "generated",
                    "arm_index": int(arm["arm_index"]),
                    "arm_id": arm_id,
                    "parameter": str(arm["parameter"]),
                    "value": str(arm["value"]),
                    "value_label": str(arm["value_label"]),
                    "plot_order": int(arm["plot_order"]),
                    "beta": float(arm["beta"]),
                    "lambda": float(arm["lam"]),
                    "u": float(arm["u"]),
                    "v": float(arm["v"]),
                    "model_seed": model_seed,
                    "run_name": str(spec["run_name"]),
                    "sampling_seed": int(sampling_seed),
                    "draw_seed": CETERIS_FINALIZATION_SEED_OFFSET + int(sampling_seed),
                    "n_samples": len(generated),
                    "bundle_path": str(bundle_path),
                    **metrics,
                }
            )
if len(ceteris_bundles) != CETERIS_EXPECTED_BUNDLE_COUNT:
    raise ValueError("Ceteris-paribus bundle loading was incomplete")
ceteris_tracking_rows = []
for arm in ceteris_arm_records:
    arm_id = str(arm["arm_id"])
    schedule = ceteris_schedules[arm_id]
    arm_index = int(arm["arm_index"])
    for level in CETERIS_TRACKING_LEVELS:
        reference_seed = (
            int(ceteris_source_config.tracking.reference_seed)
            + 1000 * arm_index
            + level
        )
        reference = ceteris_exact_noisy_marginal_samples(
            arm, schedule, level, reference_seed
        )
        reference_eval = five_param_subsample(
            reference,
            int(ceteris_source_config.tracking.n_eval),
            int(ceteris_source_config.tracking.evaluation_seed)
            + 1000 * arm_index
            + level
            + 1,
        )
        for spec in CETERIS_MODEL_SPECS:
            model_seed = int(spec["training_seed"])
            trajectory_batches = [
                ceteris_bundles[arm_id, model_seed, sampling_seed]["trajectory"][
                    level + 1
                ]
                for sampling_seed in CETERIS_SAMPLING_SEEDS
            ]
            state_eval = five_param_stratified_subsample(
                trajectory_batches,
                int(ceteris_source_config.tracking.n_eval),
                int(ceteris_source_config.tracking.evaluation_seed)
                + 1000 * arm_index
                + level
                + model_seed,
            )
            ceteris_tracking_rows.append(
                {
                    "arm_index": arm_index,
                    "arm_id": arm_id,
                    "parameter": str(arm["parameter"]),
                    "value": str(arm["value"]),
                    "model_seed": model_seed,
                    "run_name": str(spec["run_name"]),
                    "level": int(level),
                    "post_burnin": level not in CETERIS_BURN_IN_EXCLUDED_LEVELS,
                    "progress": float(schedule["progress"][level]),
                    "actual_rms": float(schedule["actual_rms"][level]),
                    "n_sampling_seeds": len(CETERIS_SAMPLING_SEEDS),
                    "n_available": sum((len(batch) for batch in trajectory_batches)),
                    "n_eval": int(ceteris_source_config.tracking.n_eval),
                    "reference_seed": reference_seed,
                    "signed_energy_distance": float(
                        energy_distance_unbiased(state_eval, reference_eval)
                    ),
                }
            )
ceteris_tracking_raw_df = (
    pd.DataFrame(ceteris_tracking_rows)
    .sort_values(["arm_index", "model_seed", "level"])
    .reset_index(drop=True)
)
ceteris_per_model_rows = []
for arm in ceteris_arm_records:
    arm_id = str(arm["arm_id"])
    for spec in CETERIS_MODEL_SPECS:
        model_seed = int(spec["training_seed"])
        generated_batches = [
            ceteris_bundles[arm_id, model_seed, sampling_seed]["final"]
            for sampling_seed in CETERIS_SAMPLING_SEEDS
        ]
        metrics = five_param_endpoint_metrics(
            generated_batches,
            ceteris_clean_reference,
            subsample=int(ceteris_source_config.evaluation.energy_subsample),
            seed=int(ceteris_source_config.evaluation.evaluation_seed),
        )
        post_burnin_tracking = ceteris_tracking_raw_df[
            (ceteris_tracking_raw_df["arm_id"] == arm_id)
            & (ceteris_tracking_raw_df["model_seed"] == model_seed)
            & ceteris_tracking_raw_df["post_burnin"]
        ]
        ceteris_per_model_rows.append(
            {
                "record_type": "generated",
                "arm_index": int(arm["arm_index"]),
                "arm_id": arm_id,
                "parameter": str(arm["parameter"]),
                "value": str(arm["value"]),
                "value_label": str(arm["value_label"]),
                "plot_order": int(arm["plot_order"]),
                "beta": float(arm["beta"]),
                "lambda": float(arm["lam"]),
                "u": float(arm["u"]),
                "v": float(arm["v"]),
                "model_seed": model_seed,
                "run_name": str(spec["run_name"]),
                "n_sampling_seeds": len(CETERIS_SAMPLING_SEEDS),
                "n_generated_samples": sum((len(batch) for batch in generated_batches)),
                **metrics,
                "post_burnin_tracking_max": float(
                    post_burnin_tracking["signed_energy_distance"].max()
                ),
                "n_tracking_checkpoints": len(post_burnin_tracking),
            }
        )
for baseline_seed_value in ceteris_source_config.evaluation.clean_baseline_seeds:
    baseline_seed = int(baseline_seed_value)
    clean_batches = []
    for batch_index, sampling_seed in enumerate(CETERIS_SAMPLING_SEEDS):
        draw_seed = baseline_seed + batch_index
        set_reproducible_seed(draw_seed)
        clean = eight_gaussians(
            int(ceteris_source_config.sampler.n_samples),
            radius=float(ceteris_source_config.data.radius),
            std=float(ceteris_source_config.data.std_clean),
            device="cpu",
        )
        clean_batches.append(clean)
        metrics = five_param_endpoint_metrics(
            clean,
            ceteris_clean_reference,
            subsample=int(ceteris_source_config.evaluation.energy_subsample),
            seed=int(ceteris_source_config.evaluation.evaluation_seed),
        )
        ceteris_raw_rows.append(
            {
                "record_type": "clean_floor",
                "arm_index": -1,
                "arm_id": "clean_floor",
                "parameter": "clean floor",
                "value": "independent draws",
                "value_label": "Independent draws",
                "plot_order": -1,
                "beta": np.nan,
                "lambda": np.nan,
                "u": np.nan,
                "v": np.nan,
                "model_seed": baseline_seed,
                "run_name": "independent_clean_draws",
                "sampling_seed": int(sampling_seed),
                "draw_seed": draw_seed,
                "n_samples": len(clean),
                "bundle_path": "",
                **metrics,
            }
        )
    metrics = five_param_endpoint_metrics(
        clean_batches,
        ceteris_clean_reference,
        subsample=int(ceteris_source_config.evaluation.energy_subsample),
        seed=int(ceteris_source_config.evaluation.evaluation_seed),
    )
    ceteris_per_model_rows.append(
        {
            "record_type": "clean_floor",
            "arm_index": -1,
            "arm_id": "clean_floor",
            "parameter": "clean floor",
            "value": "independent draws",
            "value_label": "Independent draws",
            "plot_order": -1,
            "beta": np.nan,
            "lambda": np.nan,
            "u": np.nan,
            "v": np.nan,
            "model_seed": baseline_seed,
            "run_name": "independent_clean_draws",
            "n_sampling_seeds": len(CETERIS_SAMPLING_SEEDS),
            "n_generated_samples": sum((len(batch) for batch in clean_batches)),
            **metrics,
            "post_burnin_tracking_max": np.nan,
            "n_tracking_checkpoints": 0,
        }
    )
ceteris_raw_df = (
    pd.DataFrame(ceteris_raw_rows)
    .sort_values(["arm_index", "model_seed", "sampling_seed"])
    .reset_index(drop=True)
)
ceteris_per_model_df = (
    pd.DataFrame(ceteris_per_model_rows)
    .sort_values(["arm_index", "model_seed"])
    .reset_index(drop=True)
)
ceteris_summary_rows = []
ceteris_summary_groups = [
    "record_type",
    "arm_index",
    "arm_id",
    "parameter",
    "value",
    "value_label",
    "plot_order",
    "beta",
    "lambda",
    "u",
    "v",
]
for group_values, group in ceteris_per_model_df.groupby(
    ceteris_summary_groups, sort=False, dropna=False
):
    row = dict(zip(ceteris_summary_groups, group_values))
    row["n_units"] = len(group)
    row["sampling_seeds_per_unit"] = int(group["n_sampling_seeds"].iloc[0])
    row["generated_samples_per_unit"] = int(group["n_generated_samples"].iloc[0])
    for metric in CETERIS_SUMMARY_METRICS:
        values = group[metric].dropna().to_numpy(dtype=float)
        row[f"{metric}_mean"] = float(values.mean()) if len(values) else np.nan
        row[f"{metric}_std"] = float(values.std(ddof=1)) if len(values) > 1 else np.nan
    ceteris_summary_rows.append(row)
ceteris_summary_df = (
    pd.DataFrame(ceteris_summary_rows).sort_values("arm_index").reset_index(drop=True)
)
write_csv_atomic(ceteris_raw_df, CETERIS_RAW_PATH)
write_csv_atomic(ceteris_tracking_raw_df, CETERIS_TRACKING_RAW_PATH)
write_csv_atomic(ceteris_per_model_df, CETERIS_PER_MODEL_PATH)
write_csv_atomic(ceteris_summary_df, CETERIS_SUMMARY_PATH)
write_json_atomic(
    {
        "config_artifact": str(CETERIS_CONFIG_ARTIFACT_PATH),
        "resolved_config": CETERIS_RESOLVED_CONFIG,
        "models": ceteris_result_models,
        "bundle_count": len(ceteris_bundles),
        "bundle_paths": [str(path) for path in ceteris_expected_bundle_paths],
        "pooling": "pool sampling seeds within each trained model",
        "uncertainty": "mean and sample SD across trained-model seeds",
        "energy_distance_sample_size": int(
            ceteris_source_config.evaluation.energy_subsample
        ),
        "tracking_levels": CETERIS_TRACKING_LEVELS,
        "burn_in_excluded_levels": sorted(CETERIS_BURN_IN_EXCLUDED_LEVELS),
        "tracking_sample_size": int(ceteris_source_config.tracking.n_eval),
    },
    CETERIS_PROTOCOL_PATH,
)


In [ ]:
ceteris_required_paths = [
    CETERIS_RAW_PATH,
    CETERIS_TRACKING_RAW_PATH,
    CETERIS_PER_MODEL_PATH,
    CETERIS_SUMMARY_PATH,
    CETERIS_PROTOCOL_PATH,
]
ceteris_missing_paths = [
    str(path) for path in ceteris_required_paths if not path.exists()
]
if ceteris_missing_paths:
    raise FileNotFoundError(
        f"Run the ceteris-paribus evaluation cell first; missing {ceteris_missing_paths}"
    )
ceteris_raw_df = pd.read_csv(CETERIS_RAW_PATH)
ceteris_tracking_raw_df = pd.read_csv(CETERIS_TRACKING_RAW_PATH)
ceteris_per_model_df = pd.read_csv(CETERIS_PER_MODEL_PATH)
ceteris_summary_df = pd.read_csv(CETERIS_SUMMARY_PATH)
ceteris_protocol = OmegaConf.load(CETERIS_PROTOCOL_PATH)
ceteris_protocol_resolved_config = OmegaConf.to_container(
    ceteris_protocol.resolved_config, resolve=True
)
if ceteris_protocol_resolved_config != CETERIS_RESOLVED_CONFIG:
    raise ValueError("Ceteris-paribus summaries use a different config")
if int(ceteris_protocol.bundle_count) != CETERIS_EXPECTED_BUNDLE_COUNT:
    raise ValueError("Ceteris-paribus summaries use an incomplete campaign")
ceteris_expected_row_counts = {
    "raw": CETERIS_EXPECTED_BUNDLE_COUNT
    + len(ceteris_source_config.evaluation.clean_baseline_seeds)
    * len(CETERIS_SAMPLING_SEEDS),
    "tracking_raw": len(ceteris_expected_arms)
    * len(CETERIS_MODEL_SPECS)
    * len(CETERIS_TRACKING_LEVELS),
    "per_model": len(ceteris_expected_arms) * len(CETERIS_MODEL_SPECS)
    + len(ceteris_source_config.evaluation.clean_baseline_seeds),
    "summary": len(ceteris_expected_arms) + 1,
}
ceteris_observed_row_counts = {
    "raw": len(ceteris_raw_df),
    "tracking_raw": len(ceteris_tracking_raw_df),
    "per_model": len(ceteris_per_model_df),
    "summary": len(ceteris_summary_df),
}
if ceteris_observed_row_counts != ceteris_expected_row_counts:
    raise ValueError(
        f"Unexpected ceteris-paribus summary sizes: {ceteris_observed_row_counts}"
    )
ceteris_expected_generated_arm_ids = {arm["arm_id"] for arm in ceteris_expected_arms}
ceteris_observed_generated_arm_ids = set(
    ceteris_summary_df.loc[ceteris_summary_df["record_type"] == "generated", "arm_id"]
)
if ceteris_observed_generated_arm_ids != ceteris_expected_generated_arm_ids:
    raise ValueError("Ceteris-paribus summaries contain the wrong arms")


In [ ]:
ceteris_parameter_labels = {
    "clean floor": "Clean-draw floor",
    "base": "Reference fixed path",
    "beta": r"$\beta$",
    "lambda": r"$\lambda$",
    "geometry": r"$(u,v)$",
}
ceteris_table_rows = []
for row in ceteris_summary_df.to_dict(orient="records"):
    if row["arm_id"] == "clean_floor":
        value_label = "Independent draws"
    elif row["arm_id"] == "base":
        value_label = r"$\beta=1.4,\lambda=1.8,(u,v)=(0,0.6)$"
    else:
        value_label = str(row["value_label"])
    ceteris_table_rows.append(
        {
            "Parameter": ceteris_parameter_labels[row["parameter"]],
            "Value": value_label,
            "Signed ED ": ceteris_mean_sd_entry(
                row,
                "signed_energy_distance",
                5,
            ),
            "Mode-weight TV ": ceteris_mean_sd_entry(
                row,
                "mode_weight_tv",
                4,
            ),
            "Minimum mode mass ": ceteris_mean_sd_entry(
                row,
                "minimum_mode_mass",
                4,
            ),
            "Post-burn-in tracking max ": ceteris_mean_sd_entry(
                row,
                "post_burnin_tracking_max",
                4,
            ),
        }
    )

ceteris_paribus_table = pd.DataFrame(ceteris_table_rows)
write_csv_atomic(ceteris_paribus_table, CETERIS_TABLE_PATH)


In [ ]:
ceteris_paribus_table = pd.read_csv(CETERIS_TABLE_PATH)
ceteris_paribus_table


## Path geometry, endpoint samples, and progress


In [ ]:
FIVE_PARAM_PATH_PLOT_RC = {
    **FIVE_PARAM_PLOT_RC,
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 14,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "lines.linewidth": 1.6,
}
five_param_path_arrays = {
    path_name: {
        key: path_df.sort_values("progress")[key].to_numpy()
        for key in ("progress", "actual_rms", "rho", "beta", "lambda", "u", "v")
    }
    for path_name, path_df in five_param_schedule_df.groupby("path", sort=False)
}
five_param_path_progress = five_param_path_arrays["isotropic_fixed_tail"]["progress"]
for path_arrays in five_param_path_arrays.values():
    np.testing.assert_allclose(path_arrays["progress"], five_param_path_progress)

five_param_radial_groups = [
    (
        [
            "isotropic_fixed_tail",
            "fixed_non_diagonal_fixed_tail",
            "rotating_fixed_tail",
        ],
        "Isotropic, fixed non-diagonal,\nrotating covariance",
        "0.35",
    ),
    (["joint_five_parameter"], "Joint", FIVE_PARAM_PATH_COLORS["joint_five_parameter"]),
]
five_param_curve_panels = [
    (
        (0, 0),
        "Actual RMS",
        r"$r_t$",
        [
            (
                [(name, "actual_rms") for name in FIVE_PARAM_PATH_LABELS],
                "All paths",
                "0.25",
                "-",
            )
        ],
    ),
]
for position, key, title, symbol in [
    ((0, 1), "rho", "Reference RMS", r"$\rho_t$"),
    ((0, 2), "beta", "Tail shape", r"$\beta_t$"),
    ((1, 0), "lambda", "Radial concentration", r"$\lambda_t$"),
]:
    five_param_curve_panels.append(
        (
            position,
            title,
            symbol,
            [
                ([(name, key) for name in names], label, color, "-")
                for names, label, color in five_param_radial_groups
            ],
        )
    )
five_param_curve_panels.append(
    (
        (1, 1),
        "Covariance coordinates",
        r"$u_t,\ v_t$",
        [
            (
                [
                    ("isotropic_fixed_tail", "u"),
                    ("fixed_non_diagonal_fixed_tail", "u"),
                    ("isotropic_fixed_tail", "v"),
                ],
                "$u$: both fixed paths\n$v$: isotropic",
                "0.35",
                ":",
            ),
            (
                [("fixed_non_diagonal_fixed_tail", "v")],
                "$v$: fixed non-diagonal",
                FIVE_PARAM_PATH_COLORS["fixed_non_diagonal_fixed_tail"],
                "--",
            ),
            (
                [("rotating_fixed_tail", "u"), ("joint_five_parameter", "u")],
                "$u$: rotating + joint",
                FIVE_PARAM_PATH_COLORS["joint_five_parameter"],
                "-",
            ),
            (
                [("rotating_fixed_tail", "v"), ("joint_five_parameter", "v")],
                "$v$: rotating + joint",
                FIVE_PARAM_PATH_COLORS["joint_five_parameter"],
                "--",
            ),
        ],
    )
)

with plt.rc_context(FIVE_PARAM_PATH_PLOT_RC):
    fig, axes = plt.subplots(2, 3, figsize=(9.6, 7.8), dpi=300, constrained_layout=True)
    for position, title, symbol, groups in five_param_curve_panels:
        axis = axes[position]
        for members, label, color, linestyle in groups:
            first_path, first_key = members[0]
            values = five_param_path_arrays[first_path][first_key]
            for path_name, key in members[1:]:
                np.testing.assert_allclose(
                    five_param_path_arrays[path_name][key],
                    values,
                    rtol=1e-6,
                    atol=1e-8,
                    err_msg=f"Coincident-curve group changed: {label}",
                )
            axis.plot(
                five_param_path_progress,
                values,
                color=color,
                linestyle=linestyle,
                label=label,
            )
        axis.set_title(title)
        axis.set_xlabel(r"Path progress $t$")
        axis.set_ylabel(symbol)
    axes[0, 0].set_yscale("log")
    axes[0, 1].set_yscale("log")

    for key in ("u", "v"):
        np.testing.assert_allclose(
            five_param_path_arrays["rotating_fixed_tail"][key],
            five_param_path_arrays["joint_five_parameter"][key],
            rtol=1e-6,
            atol=1e-8,
        )
    rotating = five_param_path_arrays["rotating_fixed_tail"]
    axes[1, 2].plot(
        rotating["u"],
        rotating["v"],
        color=FIVE_PARAM_PATH_COLORS["joint_five_parameter"],
        label="Rotating + joint",
    )
    for path_name, label in [
        ("isotropic_fixed_tail", "Isotropic"),
        ("fixed_non_diagonal_fixed_tail", "Fixed non-diagonal"),
    ]:
        path_arrays = five_param_path_arrays[path_name]
        for key in ("u", "v"):
            np.testing.assert_allclose(path_arrays[key], path_arrays[key][0], atol=1e-8)
        axes[1, 2].scatter(
            path_arrays["u"][0],
            path_arrays["v"][0],
            s=45,
            color=FIVE_PARAM_PATH_COLORS[path_name],
            edgecolors="white",
            linewidths=0.7,
            zorder=5,
            label=label,
        )
    axes[1, 2].set_title("Covariance path")
    axes[1, 2].set_xlabel(r"$u$")
    axes[1, 2].set_ylabel(r"$v$")
    axes[1, 2].set_aspect("equal")
    axes[1, 2].set_xlim(-0.7, 0.7)
    axes[1, 2].set_ylim(-0.7, 0.7)

    for axis in axes.flat:
        axis.grid(alpha=0.18)
        axis.legend(
            frameon=False,
            loc="upper center",
            bbox_to_anchor=(0.5, -0.28),
            borderaxespad=0,
            handlelength=1.8,
            labelspacing=0.3,
        )
    fig.savefig(
        FIVE_PARAM_OUT_DIR / "full5_spd_parameter_paths.png", bbox_inches="tight"
    )
    fig.savefig(
        FIVE_PARAM_OUT_DIR / "full5_spd_parameter_paths.pdf", bbox_inches="tight"
    )


In [ ]:
with plt.rc_context(FIVE_PARAM_PLOT_RC):
    fig, axes = plt.subplots(
        1,
        len(FIVE_PARAM_PATH_LABELS),
        figsize=(9.6, 2.55),
        dpi=300,
        constrained_layout=True,
    )
    for axis, (path_name, path_label) in zip(axes, FIVE_PARAM_PATH_LABELS.items()):
        payload = five_param_diffusion_bundles[
            (
                FIVE_PARAM_FIGURE_MODEL_SEED,
                path_name,
                FIVE_PARAM_FIGURE_SAMPLING_SEED,
            )
        ]
        samples = torch.as_tensor(payload["generated"]).detach().cpu().numpy()
        axis.scatter(
            samples[:, 0],
            samples[:, 1],
            s=1.0,
            alpha=0.35,
            color=FIVE_PARAM_PATH_COLORS[path_name],
            linewidths=0,
            rasterized=True,
        )
        axis.scatter(
            five_param_centers[:, 0],
            five_param_centers[:, 1],
            marker="x",
            s=22,
            color="black",
            linewidths=0.8,
            zorder=4,
        )
        axis.set_title(path_label)
        axis.set_xlim(-FIVE_PARAM_SCORE_LIMIT, FIVE_PARAM_SCORE_LIMIT)
        axis.set_ylim(-FIVE_PARAM_SCORE_LIMIT, FIVE_PARAM_SCORE_LIMIT)
        axis.set_aspect("equal")
        axis.set_xlabel("$x_1$")
        axis.set_ylabel("$x_2$")
    fig.savefig(
        FIVE_PARAM_OUT_DIR / "full5_spd_final_samples_by_path.png",
        bbox_inches="tight",
    )
    fig.savefig(
        FIVE_PARAM_OUT_DIR / "full5_spd_final_samples_by_path.pdf",
        bbox_inches="tight",
    )


In [ ]:
five_param_progress_targets = [0.5, 0.2, 0.08, 0.02]
five_param_progress_columns = [("Initial", 0)]
for target_rms in five_param_progress_targets:
    level = int(torch.argmin((five_param_actual_rms_reference - target_rms).abs()))
    realized_rms = float(five_param_actual_rms_reference[level])
    five_param_progress_columns.append((f"Actual RMS {realized_rms:.3f}", level + 1))
five_param_progress_columns.append(("Final", None))

with plt.rc_context(FIVE_PARAM_PLOT_RC):
    fig, axes = plt.subplots(
        len(FIVE_PARAM_PATH_LABELS),
        len(five_param_progress_columns),
        figsize=(12.0, 8.0),
        dpi=300,
        constrained_layout=True,
    )
    for row_index, (path_name, path_label) in enumerate(FIVE_PARAM_PATH_LABELS.items()):
        payload = five_param_diffusion_bundles[
            (
                FIVE_PARAM_FIGURE_MODEL_SEED,
                path_name,
                FIVE_PARAM_FIGURE_SAMPLING_SEED,
            )
        ]
        for column_index, (column_label, trajectory_index) in enumerate(
            five_param_progress_columns
        ):
            axis = axes[row_index, column_index]
            state = (
                payload["generated"]
                if trajectory_index is None
                else payload["trajectory"][trajectory_index]
            )
            state = torch.as_tensor(state).detach().cpu().numpy()
            axis.scatter(
                state[:, 0],
                state[:, 1],
                s=0.55,
                alpha=0.28,
                color=FIVE_PARAM_PATH_COLORS[path_name],
                linewidths=0,
                rasterized=True,
            )
            axis.scatter(
                five_param_centers[:, 0],
                five_param_centers[:, 1],
                marker="x",
                s=15,
                color="black",
                linewidths=0.65,
                zorder=4,
            )
            axis.set_xlim(-FIVE_PARAM_SCORE_LIMIT, FIVE_PARAM_SCORE_LIMIT)
            axis.set_ylim(-FIVE_PARAM_SCORE_LIMIT, FIVE_PARAM_SCORE_LIMIT)
            axis.set_aspect("equal")
            axis.set_xticks([])
            axis.set_yticks([])
            if row_index == 0:
                axis.set_title(column_label)
            if column_index == 0:
                axis.set_ylabel(path_label)
    fig.savefig(
        FIVE_PARAM_OUT_DIR / "full5_spd_generation_progress_by_path.png",
        bbox_inches="tight",
    )
    fig.savefig(
        FIVE_PARAM_OUT_DIR / "full5_spd_generation_progress_by_path.pdf",
        bbox_inches="tight",
    )


## Tracking the exact noisy marginal


In [ ]:
def five_param_exact_noisy_marginal_samples(
    path_name,
    level,
    n_samples,
    seed,
    *,
    device,
):
    schedule = five_param_diffusion_schedules[path_name]
    beta = float(schedule["beta"][level])
    lam = float(schedule["lam"][level])
    rho = float(schedule["rho"][level])
    dtype = torch.get_default_dtype()
    uv = schedule["uv"][level].to(device=device, dtype=dtype)
    Sigma = spd_2d_from_uv(uv)
    sigma = float(reference_rho_to_sigma(rho, beta, 2))
    set_reproducible_seed(seed)
    clean = eight_gaussians(
        n_samples,
        radius=FIVE_PARAM_DATA_RADIUS,
        std=FIVE_PARAM_STD_CLEAN,
        device=device,
    )
    noise = sample_generalized_gaussian_noise(
        n_samples,
        beta=beta,
        lam=lam,
        Sigma=Sigma,
        sigma=sigma,
        device=device,
        dtype=clean.dtype,
    )
    return clean + noise


In [ ]:
FIVE_PARAM_PATH_TRACKING_LEVELS = (
    np.linspace(
        0,
        FIVE_PARAM_DIFFUSION_N_LEVELS - 1,
        11,
    )
    .round()
    .astype(int)
    .tolist()
)
FIVE_PARAM_PATH_TRACKING_N_REFERENCE = 4096
FIVE_PARAM_PATH_TRACKING_N_EVAL = 1024
FIVE_PARAM_PATH_TRACKING_REFERENCE_SEED = 600_000
FIVE_PARAM_PATH_TRACKING_EVAL_SEED = 700_000

five_param_path_tracking_rows = []
for path_index, path_name in enumerate(FIVE_PARAM_PATH_LABELS):
    schedule = five_param_diffusion_schedules[path_name]
    for level in FIVE_PARAM_PATH_TRACKING_LEVELS:
        reference_seed = (
            FIVE_PARAM_PATH_TRACKING_REFERENCE_SEED + 1_000 * path_index + level
        )
        reference = five_param_exact_noisy_marginal_samples(
            path_name,
            level,
            FIVE_PARAM_PATH_TRACKING_N_REFERENCE,
            reference_seed,
            device="cpu",
        )
        reference_eval = five_param_subsample(
            reference,
            FIVE_PARAM_PATH_TRACKING_N_EVAL,
            FIVE_PARAM_PATH_TRACKING_EVAL_SEED + level + 1,
        )
        for spec in FIVE_PARAM_MODEL_SPECS:
            model_seed = int(spec["training_seed"])
            trajectory_batches = [
                five_param_diffusion_bundles[(model_seed, path_name, sampling_seed)][
                    "trajectory"
                ][level + 1]
                for sampling_seed in FIVE_PARAM_DIFFUSION_SEEDS
            ]
            state_eval = five_param_stratified_subsample(
                trajectory_batches,
                FIVE_PARAM_PATH_TRACKING_N_EVAL,
                FIVE_PARAM_PATH_TRACKING_EVAL_SEED + level + model_seed,
            )
            five_param_path_tracking_rows.append(
                {
                    "path": path_name,
                    "level": level,
                    "progress": float(five_param_diffusion_progress[level]),
                    "actual_rms": float(schedule["actual_rms"][level]),
                    "model_seed": model_seed,
                    "n_sampling_seeds": len(FIVE_PARAM_DIFFUSION_SEEDS),
                    "n_available": sum(len(batch) for batch in trajectory_batches),
                    "n_eval": FIVE_PARAM_PATH_TRACKING_N_EVAL,
                    "signed_energy_distance": float(
                        energy_distance_unbiased(state_eval, reference_eval)
                    ),
                }
            )

five_param_path_tracking_raw_df = pd.DataFrame(five_param_path_tracking_rows)
write_csv_atomic(
    five_param_path_tracking_raw_df,
    FIVE_PARAM_OUT_DIR / "path_tracking_raw.csv",
)
len(five_param_path_tracking_raw_df)


In [ ]:
five_param_path_tracking_raw_df = pd.read_csv(
    FIVE_PARAM_OUT_DIR / "path_tracking_raw.csv"
)
five_param_path_tracking_summary_df = five_param_path_tracking_raw_df.groupby(
    ["path", "level", "progress", "actual_rms"],
    as_index=False,
).agg(
    signed_energy_distance_mean=("signed_energy_distance", "mean"),
    signed_energy_distance_std=("signed_energy_distance", "std"),
    n_models=("model_seed", "nunique"),
)
write_csv_atomic(
    five_param_path_tracking_summary_df,
    FIVE_PARAM_OUT_DIR / "path_tracking_summary.csv",
)


In [ ]:
five_param_path_tracking_summary_df = pd.read_csv(
    FIVE_PARAM_OUT_DIR / "path_tracking_summary.csv"
)


In [ ]:
FIVE_PARAM_TRACKING_LINTHRESH = 1e-3
FIVE_PARAM_TRACKING_PLOT_RC = {
    **FIVE_PARAM_PLOT_RC,
    "font.size": 12,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 10,
}

with plt.rc_context(FIVE_PARAM_TRACKING_PLOT_RC):
    fig, axis = plt.subplots(
        1,
        1,
        figsize=(5.6, 3.5),
        dpi=300,
        constrained_layout=True,
    )
    for path_name, path_label in FIVE_PARAM_PATH_LABELS.items():
        path_df = five_param_path_tracking_summary_df[
            five_param_path_tracking_summary_df["path"] == path_name
        ].sort_values("level")
        actual_rms = path_df["actual_rms"].to_numpy()
        mean = path_df["signed_energy_distance_mean"].to_numpy()
        sd = path_df["signed_energy_distance_std"].to_numpy()
        color = FIVE_PARAM_PATH_COLORS[path_name]
        axis.plot(
            actual_rms,
            mean,
            marker="o",
            markersize=3,
            color=color,
            label=path_label,
        )
        axis.fill_between(
            actual_rms,
            mean - sd,
            mean + sd,
            color=color,
            alpha=0.15,
            linewidth=0,
        )
    axis.axhline(0.0, color="black", linewidth=0.7, alpha=0.5)
    axis.set_xscale("log")
    axis.set_yscale("symlog", linthresh=FIVE_PARAM_TRACKING_LINTHRESH)
    axis.invert_xaxis()
    axis.set_xlabel("Actual RMS")
    axis.set_ylabel("Signed energy distance\nto exact noisy marginal")
    axis.legend(
        frameon=False,
        loc="lower center",
        bbox_to_anchor=(0.5, 1.02),
        ncol=2,
        borderaxespad=0,
    )
    axis.grid(alpha=0.2)
    fig.savefig(
        FIVE_PARAM_OUT_DIR / "full5_spd_marginal_tracking.png",
        bbox_inches="tight",
    )
    fig.savefig(
        FIVE_PARAM_OUT_DIR / "full5_spd_marginal_tracking.pdf",
        bbox_inches="tight",
    )
